# 04_silver_redmar.ipynb — Limpieza REDMAR Bronze → Silver

Este notebook procesa los CSV de **Puertos del Estado / REDMAR** para generar:

```text
silver/tide_hourly/source=REDMAR/...
```

Tabla objetivo:

```text
timestamp, station_id, zona_id, lat, lon, source,
sea_level, astronomical_tide, meteorological_residual,
tide_phase, next_high_tide_time, next_low_tide_time,
hours_to_high_tide, hours_to_low_tide, daily_tidal_range
```

Notas:
- `sea_level` se conserva como variable real de REDMAR.
- `astronomical_tide` y `meteorological_residual` se rellenan si aparecen en el CSV; si no, quedan en `NaN`.
- `tide_phase`, `next_high_tide_time`, `next_low_tide_time`, `hours_to_high_tide`, `hours_to_low_tide` y `daily_tidal_range` se derivan desde la serie horaria de nivel del mar.
- No se imputan huecos en Silver. Se marcan con flags.

## Celda 0 — Montar Google Drive

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

Mounted at /content/drive


## Celda 1 — Instalar librerías necesarias

In [ ]:
!pip -q install geopandas pyarrow shapely fiona tqdm scipy

## Celda 2 — Imports, rutas y configuración

In [ ]:
from pathlib import Path
import pandas as pd
import numpy as np
import geopandas as gpd
import pyarrow as pa
import pyarrow.parquet as pq
import pyarrow.dataset as ds
import re
import unicodedata
import json
import shutil
import gc
from tqdm.auto import tqdm
from scipy.signal import find_peaks

BASE_DIR = Path("/content/drive/MyDrive/AI Projects/DeepWave Canarias")
BRONZE_DIR = BASE_DIR / "data/bronze"
SILVER_DIR = BASE_DIR / "silver"

REDMAR_DIR = BRONZE_DIR / "Puertos del Estado" / "REDMAR"
DIM_ZONE_PATH = SILVER_DIR / "beach_geography" / "beach_geography.parquet"

OUT_TIDE_DIR = SILVER_DIR / "tide_hourly"
QC_DIR = SILVER_DIR / "_quality_reports"
META_DIR = SILVER_DIR / "_metadata"

OUT_TIDE_DIR.mkdir(parents=True, exist_ok=True)
QC_DIR.mkdir(parents=True, exist_ok=True)
META_DIR.mkdir(parents=True, exist_ok=True)

SOURCE_NAME = "REDMAR"

BBOX_CANARIAS = {
    "lat_min": 27.0,
    "lat_max": 29.5,
    "lon_min": -18.5,
    "lon_max": -13.0,
}

MIN_VALID_TS = pd.Timestamp("1999-01-01", tz="UTC")
MAX_VALID_TS = pd.Timestamp("2031-01-01", tz="UTC")

print("BASE_DIR:", BASE_DIR)
print("REDMAR_DIR existe:", REDMAR_DIR.exists())
print("DIM_ZONE_PATH existe:", DIM_ZONE_PATH.exists())

if not REDMAR_DIR.exists():
    raise FileNotFoundError(f"No existe REDMAR_DIR: {REDMAR_DIR}")

if not DIM_ZONE_PATH.exists():
    raise FileNotFoundError("No existe beach_geography.parquet. Ejecuta primero 01_silver_dim_zone.ipynb.")

BASE_DIR: /content/drive/MyDrive/AI Projects/DeepWave Canarias
REDMAR_DIR existe: True
DIM_ZONE_PATH existe: True


## Celda 3 — Utilidades generales

In [ ]:
def normalize_text(value):
    if pd.isna(value):
        return np.nan

    value = str(value).strip()
    value = unicodedata.normalize("NFKD", value)
    value = "".join(c for c in value if not unicodedata.combining(c))
    value = re.sub(r"\s+", " ", value)
    return value.upper()


def normalize_col(col):
    col = normalize_text(col)
    if pd.isna(col):
        return ""
    col = re.sub(r"[^A-Z0-9]+", "_", col)
    col = re.sub(r"_+", "_", col).strip("_")
    return col


def parse_coordinate(value):
    if pd.isna(value):
        return np.nan

    if isinstance(value, (int, float, np.integer, np.floating)):
        return float(value)

    s = str(value).strip().upper()
    s = s.replace(",", ".")

    if s in ["", "NAN", "NONE", "NULL"]:
        return np.nan

    sign = 1
    if any(h in s for h in ["W", "O", "S"]):
        sign = -1
    if s.startswith("-"):
        sign = -1

    nums = re.findall(r"-?\d+(?:\.\d+)?", s)

    if not nums:
        return np.nan

    try:
        if len(nums) >= 3 and ("º" in s or "°" in s or "'" in s or '"' in s):
            deg = abs(float(nums[0]))
            minutes = float(nums[1])
            seconds = float(nums[2])
            val = deg + minutes / 60 + seconds / 3600
        elif len(nums) >= 2 and ("º" in s or "°" in s or "'" in s):
            deg = abs(float(nums[0]))
            minutes = float(nums[1])
            val = deg + minutes / 60
        else:
            val = abs(float(nums[0])) if sign == -1 else float(nums[0])

        return sign * abs(val) if sign == -1 else val

    except Exception:
        return np.nan


def to_numeric_series(series):
    s = series.astype(str).str.strip()

    missing_tokens = {
        "",
        "NA",
        "N/A",
        "NAN",
        "NULL",
        "NONE",
        "-",
        "--",
        "---",
        "S/D",
        "SD",
    }

    s = s.mask(s.str.upper().isin(missing_tokens))
    s = s.str.replace(",", ".", regex=False)
    s = s.str.replace(r"[^0-9eE+\-.]", "", regex=True)

    out = pd.to_numeric(s, errors="coerce")
    out = out.mask(out.isin([-99999, -9999, -999, 999, 9999, 99999]))

    return out


def infer_column(df, candidates):
    cols_norm = {normalize_col(col): col for col in df.columns}

    for candidate in candidates:
        candidate_norm = normalize_col(candidate)

        for col_norm, original_col in cols_norm.items():
            if candidate_norm == col_norm:
                return original_col

        for col_norm, original_col in cols_norm.items():
            if candidate_norm in col_norm:
                return original_col

    return None


def find_col_by_patterns(df, patterns, exclude_patterns=None):
    if exclude_patterns is None:
        exclude_patterns = []

    for col in df.columns:
        norm = normalize_col(col)

        if any(re.search(ex, norm) for ex in exclude_patterns):
            continue

        if any(re.search(pat, norm) for pat in patterns):
            return col

    return None


def ensure_utc(series):
    return pd.to_datetime(series, utc=True, errors="coerce")


def looks_like_date_string(value):
    s = str(value)
    return bool(
        re.search(r"\d{1,2}[/-]\d{1,2}[/-]\d{2,4}", s)
        or re.search(r"\d{4}[/-]\d{1,2}[/-]\d{1,2}", s)
        or re.search(r"\d{8,14}", s)
    )


def parse_compact_datetime_series(series):
    raw = series.astype(str).str.strip()
    raw = raw.str.replace(r"\.0$", "", regex=True)
    digits = raw.str.replace(r"\D", "", regex=True)

    candidates = [
        (14, "%Y%m%d%H%M%S"),
        (12, "%Y%m%d%H%M"),
        (10, "%Y%m%d%H"),
        (8, "%Y%m%d"),
    ]

    best_parsed = None
    best_ratio = 0

    for length, fmt in candidates:
        mask = digits.str.len() == length

        if mask.mean() < 0.5:
            continue

        parsed = pd.to_datetime(
            digits.where(mask),
            format=fmt,
            errors="coerce",
            utc=True,
        )

        ratio = parsed.notna().mean()

        if ratio > best_ratio:
            best_ratio = ratio
            best_parsed = parsed

    if best_parsed is not None and best_ratio >= 0.5:
        ts_valid = best_parsed.dropna()
        if len(ts_valid) and ts_valid.between(MIN_VALID_TS, MAX_VALID_TS).mean() >= 0.8:
            return best_parsed

    return None


def dataset_count_and_sample(path, source_name=SOURCE_NAME, sample_n=5):
    if not path.exists():
        return 0, pd.DataFrame()

    dataset = ds.dataset(str(path), format="parquet", partitioning="hive")
    count = dataset.count_rows(filter=(ds.field("source") == source_name))

    if count == 0:
        return 0, pd.DataFrame()

    sample = dataset.head(sample_n, filter=(ds.field("source") == source_name)).to_pandas()

    return count, sample

## Celda 4 — Cargar `beach_geography`

In [ ]:
beach_geography = pd.read_parquet(DIM_ZONE_PATH)

required_zone_cols = ["zona_id", "nombre_zona", "isla", "municipio", "lat", "lon"]
missing_zone_cols = [c for c in required_zone_cols if c not in beach_geography.columns]

if missing_zone_cols:
    raise ValueError(f"Faltan columnas en beach_geography: {missing_zone_cols}")

print("beach_geography shape:", beach_geography.shape)
display(beach_geography.head())

gdf_zones = gpd.GeoDataFrame(
    beach_geography.copy(),
    geometry=gpd.points_from_xy(beach_geography["lon"], beach_geography["lat"]),
    crs="EPSG:4326",
)

gdf_zones_m = gdf_zones.to_crs("EPSG:3857")

beach_geography shape: (561, 17)


,zona_id,nombre_zona,isla,municipio,lat,lon,tipo_zona,orientacion_costa,exposicion_norte,exposicion_oeste,exposicion_este,exposicion_swell_nw,exposicion_swell_ne,vulnerabilidad_costera,vulnerabilidad_source,spatial_match_isla,spatial_match_municipio
0,CAN_TF_EL_PUERTITO_0,El Puertito,Tenerife,Güímar,28.2923,-16.3766,playa,NW,1,1,0,1,1,no_disponible,/content/drive/MyDrive/AI Projects/DeepWave Ca...,True,True
1,CAN_EH_LA_RESTINGA,La Restinga,El Hierro,El Pinar de El Hierro,27.6408,-17.9799,playa,W,0,1,0,1,0,no_disponible,/content/drive/MyDrive/AI Projects/DeepWave Ca...,True,True
2,CAN_EH_ARENAS_BLANCAS,Arenas Blancas,El Hierro,Frontera,27.7667,-18.1218,playa,W,0,1,0,1,0,no_disponible,/content/drive/MyDrive/AI Projects/DeepWave Ca...,True,True
3,CAN_EH_EL_VERODAL,El Verodal,El Hierro,Frontera,27.7471,-18.1512,playa,W,0,1,0,1,0,no_disponible,/content/drive/MyDrive/AI Projects/DeepWave Ca...,True,True
4,CAN_EH_CHARCO_AZUL_0,Charco Azul,El Hierro,Frontera,27.7563,-18.0990,playa,W,0,1,0,1,0,no_disponible,/content/drive/MyDrive/AI Projects/DeepWave Ca...,True,True


## Celda 5 — Localizar archivos REDMAR

In [ ]:
redmar_files = sorted(REDMAR_DIR.glob("*.csv"))

if not redmar_files:
    raise FileNotFoundError(f"No se encontraron CSV REDMAR en {REDMAR_DIR}")

REDMAR_FILENAME_RE = re.compile(
    r"^(?P<request_id>\d+)_(?P<download_id>\d+)_(?P<station_id>\d+)_(?P<variable_group>[A-Z]+(?:_[A-Z]+)*)_(?P<start>\d{14})_(?P<end>\d{14})\.csv$",
    re.IGNORECASE,
)


def parse_redmar_filename(path):
    m = REDMAR_FILENAME_RE.match(path.name)

    if not m:
        return {
            "filename": path.name,
            "request_id": None,
            "download_id": None,
            "station_id": None,
            "variable_group": "UNKNOWN",
            "file_start": pd.NaT,
            "file_end": pd.NaT,
            "filename_parse_ok": False,
        }

    d = m.groupdict()

    return {
        "filename": path.name,
        "request_id": d["request_id"],
        "download_id": d["download_id"],
        "station_id": str(d["station_id"]),
        "variable_group": d["variable_group"].upper(),
        "file_start": pd.to_datetime(d["start"], format="%Y%m%d%H%M%S", errors="coerce", utc=True),
        "file_end": pd.to_datetime(d["end"], format="%Y%m%d%H%M%S", errors="coerce", utc=True),
        "filename_parse_ok": True,
    }


files_df = pd.DataFrame([parse_redmar_filename(p) | {"path": str(p)} for p in redmar_files])

print("Archivos REDMAR encontrados:", len(files_df))
display(files_df)

print("Conteo por variable_group:")
display(files_df["variable_group"].value_counts().reset_index())

Archivos REDMAR encontrados: 5


,filename,request_id,download_id,station_id,variable_group,file_start,file_end,filename_parse_ok,path
0,25413_52366_3450_SEA_LEVEL_20010101190120_2026...,25413,52366,3450,SEA_LEVEL,2001-01-01 19:01:20+00:00,2026-05-06 18:01:20+00:00,True,/content/drive/MyDrive/AI Projects/DeepWave Ca...
1,25413_52367_3471_SEA_LEVEL_20010101190133_2026...,25413,52367,3471,SEA_LEVEL,2001-01-01 19:01:33+00:00,2026-05-06 18:01:33+00:00,True,/content/drive/MyDrive/AI Projects/DeepWave Ca...
2,25413_52368_3463_SEA_LEVEL_20080101190136_2026...,25413,52368,3463,SEA_LEVEL,2008-01-01 19:01:36+00:00,2026-05-06 18:01:36+00:00,True,/content/drive/MyDrive/AI Projects/DeepWave Ca...
3,25413_52369_3469_SEA_LEVEL_20080101190142_2026...,25413,52369,3469,SEA_LEVEL,2008-01-01 19:01:42+00:00,2026-05-06 18:01:42+00:00,True,/content/drive/MyDrive/AI Projects/DeepWave Ca...
4,25413_52370_3459_SEA_LEVEL_20080101190146_2026...,25413,52370,3459,SEA_LEVEL,2008-01-01 19:01:46+00:00,2026-05-06 18:01:46+00:00,True,/content/drive/MyDrive/AI Projects/DeepWave Ca...


Conteo por variable_group:


,variable_group,count
0,SEA_LEVEL,5


## Celda 6 — Coordenadas fallback de estaciones REDMAR Canarias

In [ ]:
# Se usan solo si no se pueden extraer coordenadas desde la cabecera del CSV.
# Son coordenadas aproximadas del puerto/mareógrafo para permitir la asignación a zona.
# El notebook guarda coordinate_source para que quede trazable.
REDMAR_STATION_FALLBACK = {
    "3450": {
        "station_name": "Las Palmas 2",
        "lat": 28.14056,
        "lon": -15.41181,
        "isla_hint": "Gran Canaria",
    },
    "3471": {
        "station_name": "Santa Cruz de Tenerife",
        "lat": 28.4665,
        "lon": -16.2453,
        "isla_hint": "Tenerife",
    },
    "3463": {
        "station_name": "San Sebastián de La Gomera",
        "lat": 28.0904,
        "lon": -17.1108,
        "isla_hint": "La Gomera",
    },
    "3469": {
        "station_name": "Puerto del Rosario / Fuerteventura 2",
        "lat": 28.5002,
        "lon": -13.8587,
        "isla_hint": "Fuerteventura",
    },
    "3459": {
        "station_name": "La Estaca / El Hierro 2",
        "lat": 27.7856,
        "lon": -17.9007,
        "isla_hint": "El Hierro",
    },
}

## Celda 7 — Lectura robusta de CSV Puertos del Estado

In [ ]:
def read_text_lines(path, encodings=("utf-8", "utf-8-sig", "latin1", "cp1252")):
    last_error = None

    for enc in encodings:
        try:
            with open(path, "r", encoding=enc, errors="replace") as f:
                return f.readlines(), enc
        except Exception as e:
            last_error = e

    raise last_error


def detect_delimiter(lines, sample_size=120):
    candidates = [";", ",", "\t", "|"]
    scores = {sep: 0 for sep in candidates}

    for line in lines[:sample_size]:
        for sep in candidates:
            scores[sep] += line.count(sep)

    best = max(scores, key=scores.get)
    return best if scores[best] > 0 else ";"


def detect_table_start(lines, delimiter):
    date_keywords = ["FECHA", "DATE", "HORA", "TIME", "GMT", "UTC"]

    for i, line in enumerate(lines[:500]):
        norm = normalize_text(line)
        if pd.isna(norm):
            continue

        has_keyword = any(k in norm for k in date_keywords)
        has_delim = line.count(delimiter) >= 1

        if has_keyword and has_delim:
            return i

    for i, line in enumerate(lines[:500]):
        if line.count(delimiter) >= 1 and looks_like_date_string(line):
            if i > 0 and lines[i - 1].count(delimiter) >= 1 and not looks_like_date_string(lines[i - 1]):
                return i - 1
            return i

    for i, line in enumerate(lines[:500]):
        if line.count(delimiter) >= 2:
            return i

    return 0


def extract_coords_from_metadata(metadata_lines):
    lat = np.nan
    lon = np.nan
    station_name = None

    for line in metadata_lines:
        norm = normalize_text(line)
        if pd.isna(norm):
            continue

        if station_name is None and any(k in norm for k in ["MAREOGRAFO", "MAREÓGRAFO", "ESTACION", "STATION", "PUERTO"]):
            station_name = re.sub(r"\s+", " ", str(line)).strip()

        if "LAT" in norm and pd.isna(lat):
            candidate = parse_coordinate(line)
            if 20 <= candidate <= 40:
                lat = candidate

        if ("LON" in norm or "LONG" in norm) and pd.isna(lon):
            candidate = parse_coordinate(line)
            if -30 <= candidate <= 0 or 0 <= candidate <= 30:
                lon = candidate
                if lon > 0 and any(h in norm for h in [" W", " O", "OESTE", "WEST"]):
                    lon = -lon

    return lat, lon, station_name


def extract_coords_from_dataframe(df):
    lat_col = infer_column(df, ["lat", "latitude", "latitud"])
    lon_col = infer_column(df, ["lon", "lng", "longitude", "longitud"])

    lat = np.nan
    lon = np.nan

    if lat_col is not None:
        vals = df[lat_col].apply(parse_coordinate)
        vals = vals[vals.between(20, 40)]
        if len(vals):
            lat = float(vals.iloc[0])

    if lon_col is not None:
        vals = df[lon_col].apply(parse_coordinate)
        vals = vals[vals.between(-30, 0)]
        if len(vals):
            lon = float(vals.iloc[0])

    return lat, lon


def read_puertos_csv(path):
    lines, encoding = read_text_lines(path)
    delimiter = detect_delimiter(lines)
    table_start = detect_table_start(lines, delimiter)
    metadata_lines = lines[:table_start]

    read_kwargs = dict(
        sep=delimiter,
        skiprows=table_start,
        encoding=encoding,
        engine="python",
        on_bad_lines="skip",
    )

    df = pd.read_csv(path, **read_kwargs)

    # Si pandas cogió la primera fila de datos como cabecera.
    if any(looks_like_date_string(c) for c in df.columns):
        df = pd.read_csv(path, header=None, **read_kwargs)
        df.columns = [f"col_{i}" for i in range(df.shape[1])]

    df = df.dropna(axis=1, how="all")
    df.columns = [str(c).strip() for c in df.columns]

    lat_meta, lon_meta, station_name = extract_coords_from_metadata(metadata_lines)
    lat_df, lon_df = extract_coords_from_dataframe(df)

    lat = lat_meta if not pd.isna(lat_meta) else lat_df
    lon = lon_meta if not pd.isna(lon_meta) else lon_df

    meta = {
        "encoding": encoding,
        "delimiter": delimiter,
        "table_start_line": table_start,
        "metadata_line_count": len(metadata_lines),
        "lat": lat,
        "lon": lon,
        "station_name": station_name,
        "raw_columns": list(df.columns),
    }

    return df, meta

## Celda 8 — Detección temporal y de variables REDMAR

In [27]:
TIMESTAMP_EXCLUDE = [r"LAT", r"LON", r"LONG", r"POINT", r"PUNTO", r"ID", r"ESTACION"]


def is_good_timestamp(timestamp, min_valid_ratio=0.5):
    ts = pd.to_datetime(timestamp, utc=True, errors="coerce")

    if len(ts) == 0:
        return False

    if ts.notna().mean() < min_valid_ratio:
        return False

    ts_valid = ts.dropna()

    if ts_valid.empty:
        return False

    return ts_valid.between(MIN_VALID_TS, MAX_VALID_TS).mean() >= min_valid_ratio


def build_timestamp_from_filename(filename_meta, n_rows):
    start = filename_meta.get("file_start", pd.NaT)
    end = filename_meta.get("file_end", pd.NaT)

    if pd.isna(start) or pd.isna(end):
        raise ValueError(f"No se puede reconstruir timestamp desde filename: {filename_meta['filename']}")

    start = pd.to_datetime(start, utc=True).floor("min")
    end = pd.to_datetime(end, utc=True).floor("min")

    if n_rows <= 0:
        return pd.Series([], dtype="datetime64[ns, UTC]"), ["filename_fallback"]

    if n_rows == 1:
        return pd.Series([start], dtype="datetime64[ns, UTC]"), ["filename_fallback"]

    total_seconds = max((end - start).total_seconds(), n_rows - 1)
    step_seconds = total_seconds / (n_rows - 1)

    # Ajustar a frecuencias típicas de REDMAR.
    candidate_steps = np.array([60, 300, 600, 1200, 1800, 3600], dtype=float)
    nearest = candidate_steps[np.argmin(np.abs(candidate_steps - step_seconds))]

    # Si la diferencia es demasiado grande, usar interpolación temporal exacta.
    if abs(nearest - step_seconds) / max(step_seconds, 1) < 0.25:
        freq = pd.to_timedelta(int(nearest), unit="s")
        ts = pd.date_range(start=start, periods=n_rows, freq=freq, tz="UTC")
    else:
        ts = pd.to_datetime(
            np.linspace(start.value, end.value, n_rows).astype("int64"),
            utc=True,
        )

    return pd.Series(ts, dtype="datetime64[ns, UTC]"), ["filename_inferred_frequency_fallback"]


def detect_timestamp(df, filename_meta):
    date_cols = []
    time_cols = []

    for col in df.columns:
        norm = normalize_col(col)

        if any(re.search(ex, norm) for ex in TIMESTAMP_EXCLUDE):
            continue

        if "FECHA" in norm or "DATE" in norm or norm in ["DIA", "DAY"]:
            date_cols.append(col)

        if "HORA" in norm or "TIME" in norm or norm in ["HH", "H"]:
            time_cols.append(col)

    for dcol in date_cols:
        compact = parse_compact_datetime_series(df[dcol])
        if compact is not None and is_good_timestamp(compact):
            return compact, [dcol], "csv_compact_date"

        if time_cols:
            for hcol in time_cols:
                candidate = df[dcol].astype(str).str.strip() + " " + df[hcol].astype(str).str.strip()

                compact = parse_compact_datetime_series(candidate)
                if compact is not None and is_good_timestamp(compact):
                    return compact, [dcol, hcol], "csv_compact_datetime"

                parsed = pd.to_datetime(candidate, errors="coerce", dayfirst=True, utc=True)
                if is_good_timestamp(parsed):
                    return parsed, [dcol, hcol], "csv_date_time_columns"

        parsed = pd.to_datetime(df[dcol], errors="coerce", dayfirst=True, utc=True)
        if is_good_timestamp(parsed):
            return parsed, [dcol], "csv_date_column"

    for col in df.columns:
        norm = normalize_col(col)

        if any(re.search(ex, norm) for ex in TIMESTAMP_EXCLUDE):
            continue

        compact = parse_compact_datetime_series(df[col])
        if compact is not None and is_good_timestamp(compact):
            return compact, [col], "csv_compact_any_column"

    for col in df.columns:
        norm = normalize_col(col)

        if any(re.search(ex, norm) for ex in TIMESTAMP_EXCLUDE):
            continue

        parsed = pd.to_datetime(df[col], errors="coerce", dayfirst=True, utc=True)
        if is_good_timestamp(parsed):
            return parsed, [col], "csv_any_datetime_column"

    if df.shape[1] >= 2:
        candidate = df.iloc[:, 0].astype(str).str.strip() + " " + df.iloc[:, 1].astype(str).str.strip()
        parsed = pd.to_datetime(candidate, errors="coerce", dayfirst=True, utc=True)
        if is_good_timestamp(parsed):
            return parsed, [df.columns[0], df.columns[1]], "csv_first_two_columns"

    ts, cols = build_timestamp_from_filename(filename_meta, len(df))
    return ts, cols, "filename_fallback"


SEA_LEVEL_PATTERNS = [
    r"^NIVEL",
    r"NIVEL.*CM",
    r"SEA.*LEVEL",
    r"NIVEL.*MAR",
    r"NIV.*MAR",
    r"(^|_)NIV($|_)",
    r"ALTURA",
    r"SL",
]

ASTRO_TIDE_PATTERNS = [
    r"ASTRONOM",
    r"ASTRONOMICA",
    r"ASTRONOMICO",
    r"ASTRO",
    r"MAREA.*ASTRO",
    r"TIDE.*ASTRO",
]

RESIDUAL_PATTERNS = [
    r"METEOR",
    r"METEOROLOG",
    r"MAREA.*METEOR",
    r"RESID",
    r"RES_",
    r"(^|_)RES($|_)",
]


def find_redmar_value_columns(df, timestamp_cols):
    """
    Detecta columnas REDMAR evitando confundir:
    - Marea Astronómica
    - Marea Meteorológica
    """

    excludes = [
        r"^FECHA",
        r"^DATE",
        r"^HORA",
        r"^TIME",
        r"^LAT",
        r"^LON",
        r"^LONG",
        r"^ID$",
        r"^ESTACION",
        r"^STATION",
    ]

    # 1) Nivel observado
    sea_col = find_col_by_patterns(
        df,
        SEA_LEVEL_PATTERNS,
        exclude_patterns=excludes,
    )

    # 2) Marea astronómica: patrones específicos, no genéricos tipo "MAREA"
    astro_col = find_col_by_patterns(
        df,
        ASTRO_TIDE_PATTERNS,
        exclude_patterns=excludes,
    )

    # 3) Residuo / marea meteorológica
    residual_col = find_col_by_patterns(
        df,
        RESIDUAL_PATTERNS,
        exclude_patterns=excludes,
    )

    used = set(timestamp_cols)

    for c in [sea_col, astro_col, residual_col]:
        if c is not None:
            used.add(c)

    numeric_candidates = []

    for col in df.columns:
        if col in used:
            continue

        norm = normalize_col(col)

        if any(re.search(ex, norm) for ex in excludes):
            continue

        vals = to_numeric_series(df[col])

        if vals.notna().mean() > 0.5:
            numeric_candidates.append(col)

    # Fallback: si no se detectó nivel, usar la primera columna numérica.
    if sea_col is None and len(numeric_candidates) >= 1:
        sea_col = numeric_candidates[0]

    # Seguridad: no permitir que astronómica y meteorológica sean la misma columna.
    if astro_col == residual_col and astro_col is not None:
        print(
            "AVISO: astronomical_tide y meteorological_residual apuntan a la misma columna. "
            "Se fuerza astronomical_tide a NaN para evitar duplicado semántico."
        )
        astro_col = None

    return sea_col, astro_col, residual_col, numeric_candidates


def normalize_sea_level_units(series, raw_col_name):
    """
    Convierte nivel/marea a metros.

    Prioridad:
    1. Usar unidad indicada en el nombre de columna: cm, mm, m.
    2. Si no hay unidad clara, inferir por magnitud.
    3. Enmascarar códigos missing después de convertir a metros.
    """

    values = to_numeric_series(series)
    col_norm = normalize_col(raw_col_name)

    if "MM" in col_norm or "MILIM" in col_norm:
        values = values / 1000.0
        unit_inferred = "millimeters_to_meters_from_column_name"

    elif "CM" in col_norm or "CENTIM" in col_norm:
        values = values / 100.0
        unit_inferred = "centimeters_to_meters_from_column_name"

    elif re.search(r"(^|_)M($|_)", col_norm) or "METRO" in col_norm:
        unit_inferred = "meters_from_column_name"

    else:
        q99 = values.abs().quantile(0.99)

        if pd.notna(q99) and q99 > 1000:
            values = values / 1000.0
            unit_inferred = "millimeters_to_meters_by_magnitude"

        elif pd.notna(q99) and q99 > 20:
            values = values / 100.0
            unit_inferred = "centimeters_to_meters_by_magnitude"

        else:
            unit_inferred = "meters_by_magnitude"

    # Enmascarar códigos de missing una vez todo está en metros.
    # Así no eliminamos mareas astronómicas válidas tipo -1.2 m.
    values = values.mask(values <= -9)
    values = values.mask(values >= 9999)

    return values, unit_inferred

## Celda 9 — Procesar archivos REDMAR a series horarias

In [28]:
def standardize_redmar_file(path, filename_meta):
    raw_df, file_meta = read_puertos_csv(path)

    timestamp, timestamp_cols, timestamp_source = detect_timestamp(raw_df, filename_meta)

    sea_col, astro_col, residual_col, numeric_candidates = find_redmar_value_columns(
        raw_df,
        timestamp_cols=timestamp_cols,
    )

    if sea_col is None:
        raise ValueError(f"No se pudo detectar columna de nivel del mar en {path.name}")

    out = pd.DataFrame()
    out["timestamp"] = pd.to_datetime(timestamp, utc=True, errors="coerce").reset_index(drop=True)
    out["station_id"] = str(filename_meta["station_id"])
    out["source_file"] = filename_meta["filename"]

    out["sea_level"], sea_level_unit_inferred = normalize_sea_level_units(raw_df[sea_col], sea_col)

    if astro_col is not None:
        out["astronomical_tide"], astro_unit = normalize_sea_level_units(raw_df[astro_col], astro_col)
    else:
        out["astronomical_tide"] = np.nan
        astro_unit = "not_available"

    if residual_col is not None:
        out["meteorological_residual"], residual_unit = normalize_sea_level_units(raw_df[residual_col], residual_col)
    else:
        out["meteorological_residual"] = np.nan
        residual_unit = "not_available"

    out = out.dropna(subset=["timestamp"]).copy()

    invalid_time = ~out["timestamp"].between(MIN_VALID_TS, MAX_VALID_TS)
    if invalid_time.any():
        raise ValueError(
            f"{path.name} contiene timestamps fuera de rango: "
            f"{out.loc[invalid_time, 'timestamp'].min()} - {out.loc[invalid_time, 'timestamp'].max()}"
        )

    lat = file_meta.get("lat", np.nan)
    lon = file_meta.get("lon", np.nan)
    station_name = file_meta.get("station_name", None)
    coordinate_source = "csv_metadata_or_columns"

    if pd.isna(lat) or pd.isna(lon):
        fallback = REDMAR_STATION_FALLBACK.get(str(filename_meta["station_id"]))

        if fallback is not None:
            lat = fallback["lat"]
            lon = fallback["lon"]
            station_name = station_name or fallback["station_name"]
            coordinate_source = "manual_fallback_canary_redmar_station"
        else:
            coordinate_source = "missing"

    # Reducir a frecuencia horaria.
    out = (
        out
        .sort_values("timestamp")
        .drop_duplicates(subset=["timestamp"], keep="first")
        .set_index("timestamp")
    )

    numeric_cols = ["sea_level", "astronomical_tide", "meteorological_residual"]
    hourly = out[numeric_cols].resample("1h").mean()

    hourly["raw_observations_in_hour"] = out["sea_level"].resample("1h").count()
    hourly = hourly.reset_index()

    # No inventar datos: si no había observación en esa hora, sea_level queda NaN.
    hourly["station_id"] = str(filename_meta["station_id"])
    hourly["station_name"] = station_name if station_name is not None else f"REDMAR_{filename_meta['station_id']}"
    hourly["lat"] = lat
    hourly["lon"] = lon
    hourly["source_file"] = filename_meta["filename"]

    summary = {
        "filename": filename_meta["filename"],
        "station_id": str(filename_meta["station_id"]),
        "variable_group": filename_meta["variable_group"],
        "raw_rows": len(raw_df),
        "hourly_rows": len(hourly),
        "timestamp_min": hourly["timestamp"].min() if len(hourly) else pd.NaT,
        "timestamp_max": hourly["timestamp"].max() if len(hourly) else pd.NaT,
        "lat": lat,
        "lon": lon,
        "coordinate_source": coordinate_source,
        "station_name": station_name,
        "timestamp_source": timestamp_source,
        "sea_level_raw_col": sea_col,
        "astronomical_tide_raw_col": astro_col,
        "meteorological_residual_raw_col": residual_col,
        "sea_level_unit_inferred": sea_level_unit_inferred,
        "astronomical_tide_unit_inferred": astro_unit,
        "meteorological_residual_unit_inferred": residual_unit,
        "numeric_candidate_columns": json.dumps(numeric_candidates, ensure_ascii=False),
        "raw_columns": json.dumps(file_meta.get("raw_columns", []), ensure_ascii=False),
        "encoding": file_meta.get("encoding"),
        "delimiter": file_meta.get("delimiter"),
        "table_start_line": file_meta.get("table_start_line"),
    }

    return hourly, summary


processed_frames = []
file_summaries = []
read_errors = []

for _, row in tqdm(files_df.iterrows(), total=len(files_df), desc="Procesando REDMAR"):
    path = Path(row["path"])
    filename_meta = row.drop(labels=["path"]).to_dict()

    try:
        hourly, summary = standardize_redmar_file(path, filename_meta)
        processed_frames.append(hourly)
        file_summaries.append(summary)

    except Exception as e:
        read_errors.append(
            {
                "filename": path.name,
                "path": str(path),
                "error": repr(e),
            }
        )

file_summary_df = pd.DataFrame(file_summaries)
read_errors_df = pd.DataFrame(read_errors)

print("Archivos procesados correctamente:", len(file_summary_df))
print("Errores de lectura:", len(read_errors_df))

display(file_summary_df)
display(read_errors_df)

file_summary_df.to_csv(QC_DIR / "quality_redmar_file_summary.csv", index=False)
read_errors_df.to_csv(QC_DIR / "quality_redmar_read_errors.csv", index=False)

if len(read_errors_df):
    raise ValueError("Hay errores leyendo REDMAR. Revisar quality_redmar_read_errors.csv.")

if not processed_frames:
    raise ValueError("No se procesó ningún archivo REDMAR.")

redmar_hourly_raw = pd.concat(processed_frames, ignore_index=True)

print("redmar_hourly_raw shape:", redmar_hourly_raw.shape)
display(redmar_hourly_raw.head())

Procesando REDMAR:   0%|          | 0/5 [00:00<?, ?it/s]

Archivos procesados correctamente: 5
Errores de lectura: 0


,filename,station_id,variable_group,raw_rows,hourly_rows,timestamp_min,timestamp_max,lat,lon,coordinate_source,...,astronomical_tide_raw_col,meteorological_residual_raw_col,sea_level_unit_inferred,astronomical_tide_unit_inferred,meteorological_residual_unit_inferred,numeric_candidate_columns,raw_columns,encoding,delimiter,table_start_line
0,25413_52366_3450_SEA_LEVEL_20010101190120_2026...,3450,SEA_LEVEL,220411,222168,2001-01-01 00:00:00+00:00,2026-05-06 23:00:00+00:00,28.14056,-15.41181,manual_fallback_canary_redmar_station,...,Marea Astronómica (cm),Marea Meteorológica (cm),centimeters_to_meters_from_column_name,centimeters_to_meters_from_column_name,centimeters_to_meters_from_column_name,"[""Procedencia""]","[""Fecha (GMT)"", ""Nivel (cm)"", ""Marea Meteoroló...",utf-8,\t,1
1,25413_52367_3471_SEA_LEVEL_20010101190133_2026...,3471,SEA_LEVEL,222025,222168,2001-01-01 00:00:00+00:00,2026-05-06 23:00:00+00:00,28.46650,-16.24530,manual_fallback_canary_redmar_station,...,Marea Astronómica (cm),Marea Meteorológica (cm),centimeters_to_meters_from_column_name,centimeters_to_meters_from_column_name,centimeters_to_meters_from_column_name,"[""Procedencia""]","[""Fecha (GMT)"", ""Nivel (cm)"", ""Marea Meteoroló...",utf-8,\t,1
2,25413_52368_3463_SEA_LEVEL_20080101190136_2026...,3463,SEA_LEVEL,160672,160824,2008-01-01 00:00:00+00:00,2026-05-06 23:00:00+00:00,28.09040,-17.11080,manual_fallback_canary_redmar_station,...,Marea Astronómica (cm),Marea Meteorológica (cm),centimeters_to_meters_from_column_name,centimeters_to_meters_from_column_name,centimeters_to_meters_from_column_name,"[""Procedencia""]","[""Fecha (GMT)"", ""Nivel (cm)"", ""Marea Meteoroló...",utf-8,\t,1
3,25413_52369_3469_SEA_LEVEL_20080101190142_2026...,3469,SEA_LEVEL,160719,160824,2008-01-01 00:00:00+00:00,2026-05-06 23:00:00+00:00,28.50020,-13.85870,manual_fallback_canary_redmar_station,...,Marea Astronómica (cm),Marea Meteorológica (cm),centimeters_to_meters_from_column_name,centimeters_to_meters_from_column_name,centimeters_to_meters_from_column_name,"[""Procedencia""]","[""Fecha (GMT)"", ""Nivel (cm)"", ""Marea Meteoroló...",utf-8,\t,1
4,25413_52370_3459_SEA_LEVEL_20080101190146_2026...,3459,SEA_LEVEL,160753,160824,2008-01-01 00:00:00+00:00,2026-05-06 23:00:00+00:00,27.78560,-17.90070,manual_fallback_canary_redmar_station,...,Marea Astronómica (cm),Marea Meteorológica (cm),centimeters_to_meters_from_column_name,centimeters_to_meters_from_column_name,centimeters_to_meters_from_column_name,"[""Procedencia""]","[""Fecha (GMT)"", ""Nivel (cm)"", ""Marea Meteoroló...",utf-8,\t,1


""


redmar_hourly_raw shape: (926808, 10)


,timestamp,sea_level,astronomical_tide,meteorological_residual,raw_observations_in_hour,station_id,station_name,lat,lon,source_file
0,2001-01-01 00:00:00+00:00,1.106,1.120,NaN,1,3450,Las Palmas 2,28.14056,-15.41181,25413_52366_3450_SEA_LEVEL_20010101190120_2026...
1,2001-01-01 01:00:00+00:00,1.328,1.340,-0.012,1,3450,Las Palmas 2,28.14056,-15.41181,25413_52366_3450_SEA_LEVEL_20010101190120_2026...
2,2001-01-01 02:00:00+00:00,1.622,1.626,-0.004,1,3450,Las Palmas 2,28.14056,-15.41181,25413_52366_3450_SEA_LEVEL_20010101190120_2026...
3,2001-01-01 03:00:00+00:00,1.921,1.923,-0.002,1,3450,Las Palmas 2,28.14056,-15.41181,25413_52366_3450_SEA_LEVEL_20010101190120_2026...
4,2001-01-01 04:00:00+00:00,2.150,2.156,-0.006,1,3450,Las Palmas 2,28.14056,-15.41181,25413_52366_3450_SEA_LEVEL_20010101190120_2026...


## Celda 10 — Metadata de estaciones y asignación a zonas

In [29]:
station_meta = (
    redmar_hourly_raw[
        [
            "station_id",
            "station_name",
            "lat",
            "lon",
        ]
    ]
    .drop_duplicates(subset=["station_id"])
    .copy()
)

station_meta["coords_missing"] = station_meta["lat"].isna() | station_meta["lon"].isna()

station_meta["inside_bbox"] = (
    station_meta["lat"].between(BBOX_CANARIAS["lat_min"], BBOX_CANARIAS["lat_max"])
    & station_meta["lon"].between(BBOX_CANARIAS["lon_min"], BBOX_CANARIAS["lon_max"])
)

print("Estaciones REDMAR:", len(station_meta))
print("Estaciones sin coordenadas:", int(station_meta["coords_missing"].sum()))
print("Estaciones dentro bbox:", int(station_meta["inside_bbox"].sum()))

display(station_meta)

if station_meta["coords_missing"].any():
    raise ValueError("Hay estaciones REDMAR sin coordenadas. Revisar fallback o cabecera CSV.")

gdf_stations = gpd.GeoDataFrame(
    station_meta.copy(),
    geometry=gpd.points_from_xy(station_meta["lon"], station_meta["lat"]),
    crs="EPSG:4326",
)

gdf_stations_m = gdf_stations.to_crs("EPSG:3857")

nearest = gpd.sjoin_nearest(
    gdf_stations_m,
    gdf_zones_m[["zona_id", "nombre_zona", "isla", "municipio", "geometry"]],
    how="left",
    distance_col="distance_to_zona_m",
)

nearest = (
    nearest
    .sort_values("distance_to_zona_m")
    .groupby("station_id", as_index=False)
    .first()
)

station_zone = pd.DataFrame(nearest.drop(columns="geometry", errors="ignore"))
station_zone["distance_to_zona_km"] = station_zone["distance_to_zona_m"] / 1000

station_zone = station_zone[
    [
        "station_id",
        "station_name",
        "lat",
        "lon",
        "zona_id",
        "nombre_zona",
        "isla",
        "municipio",
        "distance_to_zona_km",
    ]
].copy()

display(station_zone)

station_zone.to_csv(META_DIR / "redmar_station_to_zone.csv", index=False)

print("Distancia estación REDMAR → zona más cercana, km:")
display(station_zone["distance_to_zona_km"].describe())

Estaciones REDMAR: 5
Estaciones sin coordenadas: 0
Estaciones dentro bbox: 5


,station_id,station_name,lat,lon,coords_missing,inside_bbox
0,3450,Las Palmas 2,28.14056,-15.41181,False,True
222168,3471,Santa Cruz de Tenerife,28.46650,-16.24530,False,True
444336,3463,San Sebastián de La Gomera,28.09040,-17.11080,False,True
605160,3469,Puerto del Rosario / Fuerteventura 2,28.50020,-13.85870,False,True
765984,3459,La Estaca / El Hierro 2,27.78560,-17.90070,False,True


,station_id,station_name,lat,lon,zona_id,nombre_zona,isla,municipio,distance_to_zona_km
0,3450,Las Palmas 2,28.14056,-15.41181,CAN_GC_ALCAVANERAS,Alcavaneras,Gran Canaria,Las Palmas de Gran Canaria,2.298991
1,3459,La Estaca / El Hierro 2,27.78560,-17.90070,CAN_EH_PUERTO_DE_LA_ESTACA,Puerto de la Estaca,El Hierro,Valverde,0.485897
2,3463,San Sebastián de La Gomera,28.09040,-17.11080,CAN_LG_SAN_SEBASTIAN,San Sebastián,La Gomera,San Sebastián de la Gomera,0.104402
3,3469,Puerto del Rosario / Fuerteventura 2,28.50020,-13.85870,CAN_FV_PLAYA_CHICA,Playa Chica,Fuerteventura,Puerto del Rosario,0.880881
4,3471,Santa Cruz de Tenerife,28.46650,-16.24530,CAN_TF_VALLESECO,Valleseco,Tenerife,Santa Cruz de Tenerife,2.724521


Distancia estación REDMAR → zona más cercana, km:


,distance_to_zona_km
count,5.000000
mean,1.298938
std,1.150556
min,0.104402
25%,0.485897
50%,0.880881
75%,2.298991
max,2.724521


## Celda 11 — Derivar fase de marea, pleamar/bajamar y rango diario

In [30]:
def derive_tide_features_for_station(df_station):
    df = df_station.sort_values("timestamp").copy().reset_index(drop=True)

    df["tide_phase"] = "unknown"
    df["next_high_tide_time"] = pd.NaT
    df["next_low_tide_time"] = pd.NaT
    df["hours_to_high_tide"] = np.nan
    df["hours_to_low_tide"] = np.nan
    df["daily_tidal_range"] = np.nan

    if df["sea_level"].notna().sum() < 24:
        return df

    # Fase simple por tendencia horaria.
    sea = df["sea_level"].astype(float)
    diff = sea.diff()

    df.loc[diff > 0.001, "tide_phase"] = "rising"
    df.loc[diff < -0.001, "tide_phase"] = "falling"
    df.loc[diff.abs() <= 0.001, "tide_phase"] = "stable"

    # Suavizado para detectar máximos/mínimos mareales.
    smooth = sea.interpolate(limit=3).rolling(window=5, center=True, min_periods=3).mean()

    valid = smooth.notna()

    if valid.sum() < 24:
        return df

    valid_idx = np.where(valid.values)[0]
    smooth_values = smooth.iloc[valid_idx].values

    # Marea semidiurna: distancia mínima 4 horas entre picos.
    high_rel_idx, _ = find_peaks(smooth_values, distance=4, prominence=0.02)
    low_rel_idx, _ = find_peaks(-smooth_values, distance=4, prominence=0.02)

    high_idx = valid_idx[high_rel_idx]
    low_idx = valid_idx[low_rel_idx]

    ts_ns = df["timestamp"].astype("int64").values

    high_times = df.loc[high_idx, "timestamp"].sort_values().dropna().reset_index(drop=True)
    low_times = df.loc[low_idx, "timestamp"].sort_values().dropna().reset_index(drop=True)

    if len(high_times):
        high_ns = high_times.astype("int64").values
        pos = np.searchsorted(high_ns, ts_ns, side="left")
        mask = pos < len(high_ns)

        next_high_ns = np.full(len(df), np.datetime64("NaT"), dtype="datetime64[ns]")
        next_high_ns[mask] = high_ns[pos[mask]].astype("datetime64[ns]")

        df["next_high_tide_time"] = pd.to_datetime(next_high_ns, utc=True)
        df["hours_to_high_tide"] = (
            df["next_high_tide_time"] - df["timestamp"]
        ).dt.total_seconds() / 3600

    if len(low_times):
        low_ns = low_times.astype("int64").values
        pos = np.searchsorted(low_ns, ts_ns, side="left")
        mask = pos < len(low_ns)

        next_low_ns = np.full(len(df), np.datetime64("NaT"), dtype="datetime64[ns]")
        next_low_ns[mask] = low_ns[pos[mask]].astype("datetime64[ns]")

        df["next_low_tide_time"] = pd.to_datetime(next_low_ns, utc=True)
        df["hours_to_low_tide"] = (
            df["next_low_tide_time"] - df["timestamp"]
        ).dt.total_seconds() / 3600

    # Rango diario.
    df["date"] = df["timestamp"].dt.date

    daily_range = (
        df.groupby("date")["sea_level"]
        .agg(lambda x: x.max(skipna=True) - x.min(skipna=True))
        .rename("daily_tidal_range")
    )

    df["daily_tidal_range"] = df["date"].map(daily_range)
    df = df.drop(columns=["date"])

    return df


tide_parts = []

for station_id, df_station in tqdm(redmar_hourly_raw.groupby("station_id"), desc="Derivando mareas"):
    tide_parts.append(derive_tide_features_for_station(df_station))

tide_hourly = pd.concat(tide_parts, ignore_index=True)

tide_hourly = tide_hourly.merge(
    station_zone[
        [
            "station_id",
            "zona_id",
            "nombre_zona",
            "isla",
            "municipio",
            "distance_to_zona_km",
        ]
    ],
    on="station_id",
    how="left",
)

tide_hourly["source"] = SOURCE_NAME
tide_hourly["temporal_resolution"] = "hourly"
tide_hourly["year"] = tide_hourly["timestamp"].dt.year.astype("Int64")

print("tide_hourly shape:", tide_hourly.shape)
display(tide_hourly.head())

Derivando mareas:   0%|          | 0/5 [00:00<?, ?it/s]

tide_hourly shape: (926808, 24)


,timestamp,sea_level,astronomical_tide,meteorological_residual,raw_observations_in_hour,station_id,station_name,lat,lon,source_file,...,hours_to_low_tide,daily_tidal_range,zona_id,nombre_zona,isla,municipio,distance_to_zona_km,source,temporal_resolution,year
0,2001-01-01 00:00:00+00:00,1.106,1.120,NaN,1,3450,Las Palmas 2,28.14056,-15.41181,25413_52366_3450_SEA_LEVEL_20010101190120_2026...,...,12.0,1.217,CAN_GC_ALCAVANERAS,Alcavaneras,Gran Canaria,Las Palmas de Gran Canaria,2.298991,REDMAR,hourly,2001
1,2001-01-01 01:00:00+00:00,1.328,1.340,-0.012,1,3450,Las Palmas 2,28.14056,-15.41181,25413_52366_3450_SEA_LEVEL_20010101190120_2026...,...,11.0,1.217,CAN_GC_ALCAVANERAS,Alcavaneras,Gran Canaria,Las Palmas de Gran Canaria,2.298991,REDMAR,hourly,2001
2,2001-01-01 02:00:00+00:00,1.622,1.626,-0.004,1,3450,Las Palmas 2,28.14056,-15.41181,25413_52366_3450_SEA_LEVEL_20010101190120_2026...,...,10.0,1.217,CAN_GC_ALCAVANERAS,Alcavaneras,Gran Canaria,Las Palmas de Gran Canaria,2.298991,REDMAR,hourly,2001
3,2001-01-01 03:00:00+00:00,1.921,1.923,-0.002,1,3450,Las Palmas 2,28.14056,-15.41181,25413_52366_3450_SEA_LEVEL_20010101190120_2026...,...,9.0,1.217,CAN_GC_ALCAVANERAS,Alcavaneras,Gran Canaria,Las Palmas de Gran Canaria,2.298991,REDMAR,hourly,2001
4,2001-01-01 04:00:00+00:00,2.150,2.156,-0.006,1,3450,Las Palmas 2,28.14056,-15.41181,25413_52366_3450_SEA_LEVEL_20010101190120_2026...,...,8.0,1.217,CAN_GC_ALCAVANERAS,Alcavaneras,Gran Canaria,Las Palmas de Gran Canaria,2.298991,REDMAR,hourly,2001


## Celda 12 — Flags de calidad y validaciones

In [31]:
VARIABLE_RANGES = {
    "sea_level": (-5, 5),
    "astronomical_tide": (-5, 5),
    "meteorological_residual": (-2, 2),
    "daily_tidal_range": (0, 5),
    "hours_to_high_tide": (0, 24),
    "hours_to_low_tide": (0, 24),
}


def add_quality_flags(df, variable_ranges):
    df = df.copy()

    for col, (vmin, vmax) in variable_ranges.items():
        if col not in df.columns:
            continue

        flag_col = f"{col}_flag"
        df[flag_col] = 0

        missing_mask = df[col].isna()
        outlier_mask = (~missing_mask) & ((df[col] < vmin) | (df[col] > vmax))

        df.loc[missing_mask, flag_col] = 1
        df.loc[outlier_mask, flag_col] = 2

        df[flag_col] = df[flag_col].astype("int8")

    return df


tide_hourly = add_quality_flags(tide_hourly, VARIABLE_RANGES)

required_cols = [
    "timestamp",
    "station_id",
    "zona_id",
    "lat",
    "lon",
    "source",
    "sea_level",
    "astronomical_tide",
    "meteorological_residual",
    "tide_phase",
    "next_high_tide_time",
    "next_low_tide_time",
    "hours_to_high_tide",
    "hours_to_low_tide",
    "daily_tidal_range",
    "year",
    "isla",
]

missing_cols = [c for c in required_cols if c not in tide_hourly.columns]

if missing_cols:
    raise ValueError(f"Faltan columnas requeridas en tide_hourly: {missing_cols}")

if tide_hourly.empty:
    raise ValueError("tide_hourly está vacío.")

if tide_hourly["timestamp"].isna().any():
    raise ValueError("Hay timestamps nulos en tide_hourly.")

if tide_hourly["zona_id"].isna().any():
    raise ValueError("Hay zona_id nulos en tide_hourly.")

if tide_hourly["lat"].isna().any() or tide_hourly["lon"].isna().any():
    raise ValueError("Hay coordenadas nulas en tide_hourly.")

invalid_time = ~tide_hourly["timestamp"].between(MIN_VALID_TS, MAX_VALID_TS)

if invalid_time.any():
    raise ValueError(
        "Hay timestamps fuera de rango: "
        f"{tide_hourly.loc[invalid_time, 'timestamp'].min()} - "
        f"{tide_hourly.loc[invalid_time, 'timestamp'].max()}"
    )

print("Validaciones básicas superadas.")
print("Sea level missing %:", round(tide_hourly["sea_level"].isna().mean() * 100, 3))
print("Sea level outlier %:", round((tide_hourly["sea_level_flag"] == 2).mean() * 100, 3))
display(tide_hourly.head())

Validaciones básicas superadas.
Sea level missing %: 1.6
Sea level outlier %: 0.0


,timestamp,sea_level,astronomical_tide,meteorological_residual,raw_observations_in_hour,station_id,station_name,lat,lon,source_file,...,distance_to_zona_km,source,temporal_resolution,year,sea_level_flag,astronomical_tide_flag,meteorological_residual_flag,daily_tidal_range_flag,hours_to_high_tide_flag,hours_to_low_tide_flag
0,2001-01-01 00:00:00+00:00,1.106,1.120,NaN,1,3450,Las Palmas 2,28.14056,-15.41181,25413_52366_3450_SEA_LEVEL_20010101190120_2026...,...,2.298991,REDMAR,hourly,2001,0,0,1,0,0,0
1,2001-01-01 01:00:00+00:00,1.328,1.340,-0.012,1,3450,Las Palmas 2,28.14056,-15.41181,25413_52366_3450_SEA_LEVEL_20010101190120_2026...,...,2.298991,REDMAR,hourly,2001,0,0,0,0,0,0
2,2001-01-01 02:00:00+00:00,1.622,1.626,-0.004,1,3450,Las Palmas 2,28.14056,-15.41181,25413_52366_3450_SEA_LEVEL_20010101190120_2026...,...,2.298991,REDMAR,hourly,2001,0,0,0,0,0,0
3,2001-01-01 03:00:00+00:00,1.921,1.923,-0.002,1,3450,Las Palmas 2,28.14056,-15.41181,25413_52366_3450_SEA_LEVEL_20010101190120_2026...,...,2.298991,REDMAR,hourly,2001,0,0,0,0,0,0
4,2001-01-01 04:00:00+00:00,2.150,2.156,-0.006,1,3450,Las Palmas 2,28.14056,-15.41181,25413_52366_3450_SEA_LEVEL_20010101190120_2026...,...,2.298991,REDMAR,hourly,2001,0,0,0,0,0,0


## Celda 13 — Reportes de calidad y gaps

In [32]:
def gap_report(df, station_col="station_id", timestamp_col="timestamp", expected_hours=1):
    rows = []

    for station_id, g in df[[station_col, timestamp_col]].dropna().groupby(station_col):
        ts = g[timestamp_col].sort_values().drop_duplicates()
        diffs_h = ts.diff().dropna().dt.total_seconds() / 3600
        gaps = diffs_h[diffs_h > expected_hours * 1.5]

        rows.append(
            {
                "station_id": station_id,
                "timestamp_min": ts.min(),
                "timestamp_max": ts.max(),
                "rows": len(ts),
                "gaps_count": int(len(gaps)),
                "max_gap_hours": float(gaps.max()) if len(gaps) else 0.0,
                "expected_hours": expected_hours,
            }
        )

    return pd.DataFrame(rows)


quality_summary = pd.DataFrame(
    [
        {
            "table": "tide_hourly",
            "source": SOURCE_NAME,
            "rows": len(tide_hourly),
            "unique_stations": tide_hourly["station_id"].nunique(),
            "unique_zona_id": tide_hourly["zona_id"].nunique(),
            "timestamp_min": tide_hourly["timestamp"].min(),
            "timestamp_max": tide_hourly["timestamp"].max(),
            "sea_level_missing_pct": float(tide_hourly["sea_level"].isna().mean() * 100),
            "sea_level_outlier_pct": float((tide_hourly["sea_level_flag"] == 2).mean() * 100),
            "astronomical_tide_missing_pct": float(tide_hourly["astronomical_tide"].isna().mean() * 100),
            "meteorological_residual_missing_pct": float(tide_hourly["meteorological_residual"].isna().mean() * 100),
            "hours_to_high_tide_missing_pct": float(tide_hourly["hours_to_high_tide"].isna().mean() * 100),
            "hours_to_low_tide_missing_pct": float(tide_hourly["hours_to_low_tide"].isna().mean() * 100),
        }
    ]
)

missing_by_column = (
    tide_hourly.isna()
    .mean()
    .mul(100)
    .reset_index()
    .rename(columns={"index": "column", 0: "missing_pct"})
)

gaps_by_station = gap_report(tide_hourly)

display(quality_summary)
display(missing_by_column)
display(gaps_by_station)

quality_summary.to_csv(QC_DIR / "quality_redmar_tide_summary.csv", index=False)
missing_by_column.to_csv(QC_DIR / "quality_redmar_missing_by_column.csv", index=False)
gaps_by_station.to_csv(QC_DIR / "quality_redmar_gaps_by_station.csv", index=False)

,table,source,rows,unique_stations,unique_zona_id,timestamp_min,timestamp_max,sea_level_missing_pct,sea_level_outlier_pct,astronomical_tide_missing_pct,meteorological_residual_missing_pct,hours_to_high_tide_missing_pct,hours_to_low_tide_missing_pct
0,tide_hourly,REDMAR,926808,5,5,2001-01-01 00:00:00+00:00,2026-05-06 23:00:00+00:00,1.599792,0.0,0.240395,1.957579,0.003776,0.007229


,column,missing_pct
0,timestamp,0.000000
1,sea_level,1.599792
2,astronomical_tide,0.240395
3,meteorological_residual,1.957579
4,raw_observations_in_hour,0.000000
5,station_id,0.000000
6,station_name,0.000000
7,lat,0.000000
8,lon,0.000000
9,source_file,0.000000


,station_id,timestamp_min,timestamp_max,rows,gaps_count,max_gap_hours,expected_hours
0,3450,2001-01-01 00:00:00+00:00,2026-05-06 23:00:00+00:00,222168,0,0.0,1
1,3459,2008-01-01 00:00:00+00:00,2026-05-06 23:00:00+00:00,160824,0,0.0,1
2,3463,2008-01-01 00:00:00+00:00,2026-05-06 23:00:00+00:00,160824,0,0.0,1
3,3469,2008-01-01 00:00:00+00:00,2026-05-06 23:00:00+00:00,160824,0,0.0,1
4,3471,2001-01-01 00:00:00+00:00,2026-05-06 23:00:00+00:00,222168,0,0.0,1


## Celda 14 — Guardar Parquet particionado

In [33]:
def remove_existing_source_partition(base_dir, source_name=SOURCE_NAME):
    source_path = base_dir / f"source={source_name}"
    if source_path.exists():
        shutil.rmtree(source_path)
        print("Eliminada partición antigua:", source_path)


def write_partitioned_parquet(df, base_dir):
    df = df.copy()

    df["source"] = df["source"].fillna(SOURCE_NAME).astype(str)
    df["isla"] = df["isla"].fillna("ISLA_DESCONOCIDA").astype(str)
    df["year"] = df["year"].astype("int64")

    table = pa.Table.from_pandas(df, preserve_index=False)

    pq.write_to_dataset(
        table,
        root_path=str(base_dir),
        partition_cols=["source", "year", "isla"],
        compression="snappy",
    )


final_cols = [
    "timestamp",
    "station_id",
    "station_name",
    "zona_id",
    "lat",
    "lon",
    "source",
    "sea_level",
    "astronomical_tide",
    "meteorological_residual",
    "tide_phase",
    "next_high_tide_time",
    "next_low_tide_time",
    "hours_to_high_tide",
    "hours_to_low_tide",
    "daily_tidal_range",
    "distance_to_zona_km",
    "raw_observations_in_hour",
    "temporal_resolution",
    "year",
    "isla",
    "sea_level_flag",
    "astronomical_tide_flag",
    "meteorological_residual_flag",
    "daily_tidal_range_flag",
    "hours_to_high_tide_flag",
    "hours_to_low_tide_flag",
    "source_file",
]

for col in final_cols:
    if col not in tide_hourly.columns:
        tide_hourly[col] = np.nan

tide_hourly_final = tide_hourly[final_cols].copy()

remove_existing_source_partition(OUT_TIDE_DIR, SOURCE_NAME)
write_partitioned_parquet(tide_hourly_final, OUT_TIDE_DIR)

print("Guardado REDMAR en:")
print(OUT_TIDE_DIR / f"source={SOURCE_NAME}")

Eliminada partición antigua: /content/drive/MyDrive/AI Projects/DeepWave Canarias/silver/tide_hourly/source=REDMAR
Guardado REDMAR en:
/content/drive/MyDrive/AI Projects/DeepWave Canarias/silver/tide_hourly/source=REDMAR


## Celda 15 — Comprobación final de lectura

In [34]:
tide_count, tide_sample = dataset_count_and_sample(OUT_TIDE_DIR)

print("Filas guardadas tide_hourly REDMAR:", tide_count)

if len(tide_sample):
    display(tide_sample)

if tide_count == 0:
    raise ValueError("No se guardó ningún registro REDMAR.")

global_summary = pd.DataFrame(
    [
        {
            "table": "tide_hourly",
            "source": SOURCE_NAME,
            "rows": tide_count,
            "stations": tide_hourly_final["station_id"].nunique(),
            "timestamp_min": tide_hourly_final["timestamp"].min(),
            "timestamp_max": tide_hourly_final["timestamp"].max(),
            "sea_level_missing_pct": float(tide_hourly_final["sea_level"].isna().mean() * 100),
            "sea_level_outlier_pct": float((tide_hourly_final["sea_level_flag"] == 2).mean() * 100),
        }
    ]
)

display(global_summary)

global_summary.to_csv(QC_DIR / "quality_redmar_global_summary.csv", index=False)

print("Reportes REDMAR:")
for p in sorted(QC_DIR.glob("quality_redmar*.csv")):
    print("-", p)

print("\nMetadatos REDMAR:")
for p in sorted(META_DIR.glob("redmar*.csv")):
    print("-", p)

print("\nValidación final REDMAR superada.")

Filas guardadas tide_hourly REDMAR: 926808


,timestamp,station_id,station_name,zona_id,lat,lon,sea_level,astronomical_tide,meteorological_residual,tide_phase,...,sea_level_flag,astronomical_tide_flag,meteorological_residual_flag,daily_tidal_range_flag,hours_to_high_tide_flag,hours_to_low_tide_flag,source_file,source,year,isla
0,2001-01-01 00:00:00+00:00,3450,Las Palmas 2,CAN_GC_ALCAVANERAS,28.14056,-15.41181,1.106,1.120,NaN,unknown,...,0,0,1,0,0,0,25413_52366_3450_SEA_LEVEL_20010101190120_2026...,REDMAR,2001,Gran Canaria
1,2001-01-01 01:00:00+00:00,3450,Las Palmas 2,CAN_GC_ALCAVANERAS,28.14056,-15.41181,1.328,1.340,-0.012,rising,...,0,0,0,0,0,0,25413_52366_3450_SEA_LEVEL_20010101190120_2026...,REDMAR,2001,Gran Canaria
2,2001-01-01 02:00:00+00:00,3450,Las Palmas 2,CAN_GC_ALCAVANERAS,28.14056,-15.41181,1.622,1.626,-0.004,rising,...,0,0,0,0,0,0,25413_52366_3450_SEA_LEVEL_20010101190120_2026...,REDMAR,2001,Gran Canaria
3,2001-01-01 03:00:00+00:00,3450,Las Palmas 2,CAN_GC_ALCAVANERAS,28.14056,-15.41181,1.921,1.923,-0.002,rising,...,0,0,0,0,0,0,25413_52366_3450_SEA_LEVEL_20010101190120_2026...,REDMAR,2001,Gran Canaria
4,2001-01-01 04:00:00+00:00,3450,Las Palmas 2,CAN_GC_ALCAVANERAS,28.14056,-15.41181,2.150,2.156,-0.006,rising,...,0,0,0,0,0,0,25413_52366_3450_SEA_LEVEL_20010101190120_2026...,REDMAR,2001,Gran Canaria


,table,source,rows,stations,timestamp_min,timestamp_max,sea_level_missing_pct,sea_level_outlier_pct
0,tide_hourly,REDMAR,926808,5,2001-01-01 00:00:00+00:00,2026-05-06 23:00:00+00:00,1.599792,0.0


Reportes REDMAR:
- /content/drive/MyDrive/AI Projects/DeepWave Canarias/silver/_quality_reports/quality_redmar_file_summary.csv
- /content/drive/MyDrive/AI Projects/DeepWave Canarias/silver/_quality_reports/quality_redmar_gaps_by_station.csv
- /content/drive/MyDrive/AI Projects/DeepWave Canarias/silver/_quality_reports/quality_redmar_global_summary.csv
- /content/drive/MyDrive/AI Projects/DeepWave Canarias/silver/_quality_reports/quality_redmar_missing_by_column.csv
- /content/drive/MyDrive/AI Projects/DeepWave Canarias/silver/_quality_reports/quality_redmar_read_errors.csv
- /content/drive/MyDrive/AI Projects/DeepWave Canarias/silver/_quality_reports/quality_redmar_tide_summary.csv

Metadatos REDMAR:
- /content/drive/MyDrive/AI Projects/DeepWave Canarias/silver/_metadata/redmar_station_to_zone.csv

Validación final REDMAR superada.


In [35]:
print("=== COMPROBACIÓN FINAL REDMAR ===")

print("\n1) Archivos procesados:")
print("file_summary_df shape:", file_summary_df.shape)
print("read_errors_df shape:", read_errors_df.shape)

if len(read_errors_df):
    print("\nERRORES:")
    display(read_errors_df)

print("\n2) tide_hourly_final:")
print("shape:", tide_hourly_final.shape)
print("estaciones:", tide_hourly_final["station_id"].nunique())
print("zonas:", tide_hourly_final["zona_id"].nunique())
print("timestamp_min:", tide_hourly_final["timestamp"].min())
print("timestamp_max:", tide_hourly_final["timestamp"].max())

print("\n3) Nulos principales:")
cols_check = [
    "timestamp",
    "station_id",
    "zona_id",
    "lat",
    "lon",
    "sea_level",
    "tide_phase",
    "daily_tidal_range",
    "hours_to_high_tide",
    "hours_to_low_tide",
]

display(
    tide_hourly_final[cols_check]
    .isna()
    .mean()
    .mul(100)
    .reset_index()
    .rename(columns={"index": "column", 0: "missing_pct"})
)

print("\n4) Flags principales:")
for col in [
    "sea_level_flag",
    "daily_tidal_range_flag",
    "hours_to_high_tide_flag",
    "hours_to_low_tide_flag",
]:
    if col in tide_hourly_final.columns:
        print(col)
        display(tide_hourly_final[col].value_counts(normalize=True).mul(100).reset_index())

print("\n5) Parquet guardado:")
tide_count, tide_sample = dataset_count_and_sample(OUT_TIDE_DIR)

print("Filas guardadas tide_hourly REDMAR:", tide_count)

if len(tide_sample):
    display(tide_sample)

print("\n6) Estaciones → zonas:")
display(station_zone)

print("\n7) Distancias estación REDMAR → zona:")
display(station_zone["distance_to_zona_km"].describe())

if len(read_errors_df) > 0:
    raise ValueError("Hay errores de lectura REDMAR.")

if tide_hourly_final.empty:
    raise ValueError("tide_hourly_final está vacío.")

if tide_count == 0:
    raise ValueError("No se guardó ningún registro REDMAR.")

if tide_hourly_final["zona_id"].isna().any():
    raise ValueError("Hay zona_id nulos.")

if tide_hourly_final["lat"].isna().any() or tide_hourly_final["lon"].isna().any():
    raise ValueError("Hay coordenadas nulas.")

if tide_hourly_final["sea_level"].isna().mean() > 0.25:
    raise ValueError("Más del 25% de sea_level está nulo. Revisar lectura/resampleo.")

print("\n✅ REDMAR parece correcto para Silver.")

=== COMPROBACIÓN FINAL REDMAR ===

1) Archivos procesados:
file_summary_df shape: (5, 23)
read_errors_df shape: (0, 0)

2) tide_hourly_final:
shape: (926808, 28)
estaciones: 5
zonas: 5
timestamp_min: 2001-01-01 00:00:00+00:00
timestamp_max: 2026-05-06 23:00:00+00:00

3) Nulos principales:


,column,missing_pct
0,timestamp,0.000000
1,station_id,0.000000
2,zona_id,0.000000
3,lat,0.000000
4,lon,0.000000
5,sea_level,1.599792
6,tide_phase,0.000000
7,daily_tidal_range,1.255924
8,hours_to_high_tide,0.003776
9,hours_to_low_tide,0.007229



4) Flags principales:
sea_level_flag


,sea_level_flag,proportion
0,0,98.400208
1,1,1.599792


daily_tidal_range_flag


...


6) Estaciones → zonas:


,station_id,station_name,lat,lon,zona_id,nombre_zona,isla,municipio,distance_to_zona_km
0,3450,Las Palmas 2,28.14056,-15.41181,CAN_GC_ALCAVANERAS,Alcavaneras,Gran Canaria,Las Palmas de Gran Canaria,2.298991
1,3459,La Estaca / El Hierro 2,27.78560,-17.90070,CAN_EH_PUERTO_DE_LA_ESTACA,Puerto de la Estaca,El Hierro,Valverde,0.485897
2,3463,San Sebastián de La Gomera,28.09040,-17.11080,CAN_LG_SAN_SEBASTIAN,San Sebastián,La Gomera,San Sebastián de la Gomera,0.104402
3,3469,Puerto del Rosario / Fuerteventura 2,28.50020,-13.85870,CAN_FV_PLAYA_CHICA,Playa Chica,Fuerteventura,Puerto del Rosario,0.880881
4,3471,Santa Cruz de Tenerife,28.46650,-16.24530,CAN_TF_VALLESECO,Valleseco,Tenerife,Santa Cruz de Tenerife,2.724521



7) Distancias estación REDMAR → zona:


,distance_to_zona_km
count,5.000000
mean,1.298938
std,1.150556
min,0.104402
25%,0.485897
50%,0.880881
75%,2.298991
max,2.724521



✅ REDMAR parece correcto para Silver.


## Resultado esperado

Al terminar deberían existir:

```text
silver/tide_hourly/source=REDMAR/year=YYYY/isla=.../*.parquet
silver/_quality_reports/quality_redmar_tide_summary.csv
silver/_quality_reports/quality_redmar_global_summary.csv
silver/_quality_reports/quality_redmar_missing_by_column.csv
silver/_quality_reports/quality_redmar_gaps_by_station.csv
silver/_metadata/redmar_station_to_zone.csv
```

Antes de continuar, comprueba:

```text
Archivos procesados correctamente = número de CSV REDMAR
Errores de lectura = 0
Filas guardadas tide_hourly REDMAR > 0
Validación final REDMAR superada
```

Si `astronomical_tide` o `meteorological_residual` salen con muchos nulos, no es necesariamente error: algunos CSV descargados como `SEA_LEVEL` solo traen nivel observado.